# EU AI Act RAG System - Complete Overview

This notebook demonstrates the complete RAG system for querying the EU AI Act.

## System Components:
1. Document Ingestion (561 chunks)
2. Multi-Provider Embeddings (WatsonX Granite)
3. Hybrid Retrieval (FAISS + BM25 + RRF)
4. LLM Generation with Citations
5. Evaluation Framework

In [ ]:
import sys
import pickle
import json
import faiss
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv()

from src.embeddings.multi_provider_generator import MultiProviderEmbeddingGenerator
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.sparse_retriever import SparseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever

print("✓ Imports successful")

## 1. Load Processed Data

In [ ]:
chunks_path = Path("../data/processed/chunks.json")
with open(chunks_path, 'r') as f:
    chunks_data = json.load(f)

print(f"Total chunks: {len(chunks_data)}")
print(f"\nFirst chunk:")
print(f"ID: {chunks_data[0]['chunk_id']}")
print(f"Type: {chunks_data[0]['metadata']['doc_type']}")
print(f"Pages: {chunks_data[0]['metadata']['page_numbers']}")
print(f"Text preview: {chunks_data[0]['text'][:200]}...")

## 2. Initialize Embedding Generator

In [ ]:
embedding_generator = MultiProviderEmbeddingGenerator()

test_text = "What are the prohibited AI practices?"
test_embedding = embedding_generator.generate_embeddings([test_text])[0]

print(f"Embedding dimension: {len(test_embedding)}")
print(f"Embedding type: {type(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

## 3. Load FAISS Index

In [ ]:
chunks_pkl_path = Path("../data/indexes/chunks.pkl")
with open(chunks_pkl_path, 'rb') as f:
    chunks = pickle.load(f)

index_path = Path("../data/indexes/faiss_index.bin")
index = faiss.read_index(str(index_path))

print(f"FAISS index type: {type(index)}")
print(f"Total vectors: {index.ntotal}")
print(f"Dimension: 768")
print(f"Chunks loaded: {len(chunks)}")

## 4. Initialize Hybrid Retriever

In [ ]:
dense_retriever = DenseRetriever(
    index=index,
    chunks=chunks,
    embedding_generator=embedding_generator
)

sparse_retriever = SparseRetriever(
    chunks=chunks,
    k1=1.5,
    b=0.75
)

hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=sparse_retriever,
    fusion_method='rrf',
    rrf_k=60,
    alpha=0.5
)

print("✓ Hybrid retriever initialized")

## 5. Test Query

In [ ]:
query = "What are the prohibited AI practices according to the EU AI Act?"

results = hybrid_retriever.retrieve(
    query=query,
    top_k=5,
    dense_k=20,
    sparse_k=20,
    dense_weight=0.5,
    sparse_weight=0.5
)

print(f"Query: {query}\n")
print(f"Retrieved {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. {result.chunk_id} (Score: {result.score:.4f})")
    print(f"   Pages: {result.metadata.get('page_numbers', 'N/A')}")
    print(f"   Type: {result.metadata.get('doc_type', 'N/A')}")
    print(f"   Text: {result.text[:150]}...\n")

## 6. Load Evaluation Results

In [ ]:
results_path = Path("../results/all_queries_evaluation.json")
with open(results_path, 'r') as f:
    eval_results = json.load(f)

summary = eval_results['summary']
print("Evaluation Summary:")
print(f"Total Queries: {summary['total_queries']}")
print(f"Successful: {summary['successful']}")
print(f"Success Rate: {summary['success_rate']:.1f}%")
print(f"Average Latency: {summary['avg_latency_ms']:.2f}ms")

## 7. System Statistics

In [ ]:
print("System Configuration:")
print(f"- Document: EU AI Act (Regulation 2024/1689)")
print(f"- Total Chunks: {len(chunks)}")
print(f"- Embedding Model: WatsonX granite-278m-multilingual")
print(f"- Embedding Dimension: 768")
print(f"- Vector Database: FAISS IndexHNSWFlat")
print(f"- Sparse Retrieval: BM25 (k1=1.5, b=0.75)")
print(f"- Fusion Method: RRF (k=60)")
print(f"- Chunk Size: 512 tokens")
print(f"- Chunk Overlap: 50 tokens")